In [ ]:
import asyncio
from typing import AsyncGenerator
from google.adk.agents import LoopAgent, LlmAgent, BaseAgent
from google.adk.events import Event, EventActions
from google.adk.agents.invocation_context import InvocationContext

# 最佳实践：将自定义代理定义为完整的、自描述的类。
class ConditionChecker(BaseAgent):
    """A custom agent that checks for a 'completed' status in the session state."""
    name: str = "ConditionChecker"
    description: str = "Checks if a process is complete and signals the loop to stop."

    async def _run_async_impl(
        self, context: InvocationContext
    ) -> AsyncGenerator[Event, None]:
        """Checks state and yields an event to either continue or stop the loop."""
        status = context.session.state.get("status", "pending")
        is_done = (status == "completed")

        if is_done:
            # 当条件满足时升级以终止循环。
            yield Event(author=self.name, actions=EventActions(escalate=True))
        else:
            # 产生一个简单的事件来继续循环。
            yield Event(author=self.name, content="Condition not met, continuing loop.")

# 更正：LlmAgent 必须有型号和明确的说明。
process_step = LlmAgent(
    name="ProcessingStep",
    model="gemini-2.0-flash-exp",
    instruction="You are a step in a longer process. Perform your task. If you are the final step, update session state by setting 'status' to 'completed'."
)

# LoopAgent 协调工作流程。
poller = LoopAgent(
    name="StatusPoller",
    max_iterations=10,
    sub_agents=[
        process_step,
        ConditionChecker() # 实例化定义良好的自定义代理。
    ]
)

# 该轮询器现在将执行“process_step”，然后执行“ConditionChecker”
# 重复直到状态为“完成”或已经经过 10 次迭代。